# Inference — Custom Images

Corre el modelo VAE sobre imágenes propias (sin ground truth).

## Naming convention

Pon las imágenes en la carpeta `test_images/` con el formato:
```
test_images/
  hazelnut_muestra1.png
  hazelnut_screenshot.png
  bottle_foto.png
  ...
```
El prefijo antes del primer `_` debe ser una clase MVTec válida:
`bottle, cable, capsule, carpet, grid, hazelnut, leather,
metal_nut, pill, screw, tile, toothbrush, transistor, wood, zipper`

## Qué esperar

- El modelo fue entrenado con **un objeto centrado** por imagen (estilo MVTec).
- Imágenes con múltiples objetos, fondos distintos o composición diferente
  generarán mapas más ruidosos — pero el modelo igual intentará reconstruir
  lo que reconoce y marcará lo que no pudo reconstruir.
- Sin ground truth → no hay F1, solo visualización.
- El threshold usado es el calibrado en el test set de MVTec para esa clase.

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import cv2
from pathlib import Path
from PIL import Image
import torchvision.transforms as T

from src.model    import ConvVAE
from src.evaluate import anomaly_map, apply_morphology

# ─── Configuración ───────────────────────────────────────────
CKPT_DIR   = Path("checkpoints_v2")   # usa V2 por defecto (mejor resultado)
IMG_SIZE   = 256
LATENT_DIM = 128
LAMBDA1    = 0.5
LAMBDA2    = 0.5
INPUT_DIR  = Path("test_images")      # pon tus imágenes aquí

# Threshold por clase — calibrado en el test set MVTec (run V2)
# Si no tienes V2 y usas checkpoints/ baseline, estos valores son orientativos
THRESHOLDS = {
    "bottle":     0.469,
    "cable":      0.551,
    "capsule":    0.265,
    "carpet":     0.000,
    "grid":       0.857,
    "hazelnut":   0.449,
    "leather":    0.551,
    "metal_nut":  0.204,
    "pill":       0.327,
    "screw":      0.673,
    "tile":       0.000,
    "toothbrush": 0.551,
    "transistor": 0.224,
    "wood":       0.633,
    "zipper":     0.469,
}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
INPUT_DIR.mkdir(exist_ok=True)
print(f"Pon tus imágenes en: {INPUT_DIR.resolve()}")

In [ ]:
# ─── Helpers ─────────────────────────────────────────────────

def load_model(category: str) -> ConvVAE:
    """Carga el checkpoint V2 para la categoría indicada."""
    ckpt = CKPT_DIR / f"{category}.pth"
    if not ckpt.exists():
        raise FileNotFoundError(
            f"Checkpoint no encontrado: {ckpt}\n"
            f"Descarga los checkpoints del Drive y colócalos en {CKPT_DIR}/"
        )
    model = ConvVAE(latent_dim=LATENT_DIM, img_size=IMG_SIZE).to(device)
    model.load_state_dict(torch.load(ckpt, map_location=device, weights_only=False))
    model.eval()
    print(f"  Modelo cargado: {ckpt}")
    return model


def preprocess(img_path: Path) -> torch.Tensor:
    """Carga y preprocesa imagen: resize 256x256, ToTensor [0,1]."""
    transform = T.Compose([
        T.Resize((IMG_SIZE, IMG_SIZE)),
        T.ToTensor(),
    ])
    img = Image.open(img_path).convert("RGB")
    return transform(img).unsqueeze(0)   # [1, 3, 256, 256]


def infer(img_path: Path, model: ConvVAE, threshold: float):
    """
    Corre el pipeline completo sobre una imagen:
      1. Reconstrucción VAE
      2. Anomaly map (L2 + multi-scale SSIM per-pixel)
      3. Normalización min-max local (no hay test set → normalización por imagen)
      4. Threshold + morfología → máscara binaria
    
    Nota: sin test set completo, la normalización es por imagen (no global).
    El threshold puede no ser directamente comparable al calibrado en MVTec.
    """
    img_t = preprocess(img_path)

    # Reconstrucción
    with torch.no_grad():
        recon_t = model.reconstruct(img_t.to(device))

    # Anomaly map (crudo, sin normalizar)
    raw_map = anomaly_map(model, img_t, device, LAMBDA1, LAMBDA2, sigma=4.0)

    # Normalización local (min-max por imagen)
    lo, hi = raw_map.min(), raw_map.max()
    norm_map = (raw_map - lo) / (hi - lo + 1e-8)

    # Threshold + morfología
    pred_bin = (norm_map >= threshold).astype(np.uint8) * 255
    pred_bin = apply_morphology(pred_bin, kernel_size=5)

    # numpy para visualizar
    img_np   = img_t.squeeze().permute(1, 2, 0).numpy()
    recon_np = recon_t.squeeze().permute(1, 2, 0).cpu().numpy()

    return img_np, recon_np, norm_map, pred_bin


def parse_category(img_path: Path) -> str:
    """Extrae la categoría del nombre del archivo: 'hazelnut_foto.png' → 'hazelnut'."""
    name = img_path.stem          # sin extensión
    category = name.split("_")[0] # todo antes del primer _
    valid = set(THRESHOLDS.keys())
    if category not in valid:
        raise ValueError(
            f"Categoría '{category}' no reconocida en '{img_path.name}'.\n"
            f"El archivo debe empezar con una clase MVTec válida: {sorted(valid)}"
        )
    return category


print("Helpers cargados.")

In [ ]:
# ─── Inferencia sobre todas las imágenes en test_images/ ─────

images = sorted(INPUT_DIR.glob("*.png")) + sorted(INPUT_DIR.glob("*.jpg"))

if not images:
    print(f"No se encontraron imágenes en {INPUT_DIR}/")
    print("Agrega imágenes con formato: hazelnut_foto.png, bottle_test.jpg, etc.")
else:
    print(f"{len(images)} imagen(es) encontrada(s):\n")
    models_cache = {}  # cache para no recargar el mismo modelo dos veces

    for img_path in images:
        print(f"Procesando: {img_path.name}")
        try:
            category  = parse_category(img_path)
            threshold = THRESHOLDS[category]

            # Cargar modelo (cache)
            if category not in models_cache:
                models_cache[category] = load_model(category)
            model = models_cache[category]

            # Inferencia
            img_np, recon_np, norm_map, pred_bin = infer(img_path, model, threshold)

            # Plot
            fig, axes = plt.subplots(1, 4, figsize=(16, 4))
            fig.suptitle(f"{img_path.name}  |  clase: {category}  |  threshold: {threshold:.3f}", fontsize=11)

            axes[0].imshow(img_np);                  axes[0].set_title("Input");          axes[0].axis("off")
            axes[1].imshow(recon_np);                axes[1].set_title("Reconstruction"); axes[1].axis("off")
            axes[2].imshow(norm_map, cmap="hot");    axes[2].set_title("Anomaly Map");    axes[2].axis("off")
            axes[3].imshow(pred_bin, cmap="gray");   axes[3].set_title("Predicted Mask"); axes[3].axis("off")

            plt.tight_layout()
            plt.show()

        except (FileNotFoundError, ValueError) as e:
            print(f"  ERROR: {e}\n")

In [ ]:
# ─── Inferencia sobre UNA imagen específica ──────────────────
# Útil para probar sin mover archivos al folder

SINGLE_IMAGE = Path(r"C:\Users\dguer\Documents\ShareX\Screenshots\2026-05\Discord_fdbp13ov9A.png")
SINGLE_CATEGORY = "hazelnut"   # especifica la clase manualmente
SINGLE_THRESHOLD = THRESHOLDS[SINGLE_CATEGORY]

if SINGLE_IMAGE.exists():
    print(f"Imagen: {SINGLE_IMAGE.name}")
    print(f"Clase: {SINGLE_CATEGORY} | threshold: {SINGLE_THRESHOLD}")

    if SINGLE_CATEGORY not in models_cache:
        models_cache[SINGLE_CATEGORY] = load_model(SINGLE_CATEGORY)
    model = models_cache[SINGLE_CATEGORY]

    img_np, recon_np, norm_map, pred_bin = infer(SINGLE_IMAGE, model, SINGLE_THRESHOLD)

    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    fig.suptitle(f"{SINGLE_IMAGE.name}  |  {SINGLE_CATEGORY}  |  threshold: {SINGLE_THRESHOLD:.3f}", fontsize=11)

    axes[0].imshow(img_np);               axes[0].set_title("Input");          axes[0].axis("off")
    axes[1].imshow(recon_np);             axes[1].set_title("Reconstruction"); axes[1].axis("off")
    axes[2].imshow(norm_map, cmap="hot"); axes[2].set_title("Anomaly Map");    axes[2].axis("off")
    axes[3].imshow(pred_bin, cmap="gray");axes[3].set_title("Predicted Mask"); axes[3].axis("off")

    plt.tight_layout()
    plt.show()
else:
    print(f"No se encontró: {SINGLE_IMAGE}")
    print("Cambia SINGLE_IMAGE al path de tu imagen.")